# Appendix 10.5: Extended Thinking

- [Lesson](#lesson)
- [Exercise](#exercise)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `think` helper function.

In [ ]:
!pip install anthropic

import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def think(prompt, budget_tokens=2000, system_prompt=""):
    # Note: max_tokens must be greater than budget_tokens, since max_tokens covers
    # both the thinking block and the final answer.
    return client.messages.create(
        model=MODEL_NAME,
        max_tokens=budget_tokens + 1000,
        system=system_prompt,
        thinking={"type": "enabled", "budget_tokens": budget_tokens},
        messages=[{"role": "user", "content": prompt}],
    )

---

## Lesson

Recall **Chapter 6**: asking Claude to "think step by step" in its output before answering often turns a wrong answer into a right one. That's *manual* chain-of-thought — you're engineering the prompt to make the model externalize reasoning in its normal response text.

**Extended thinking** is a different, native capability: reasoning-enabled models can produce a separate **thinking block** before their answer, using reasoning learned through training (not just prompted into following steps you wrote). The model works through the problem, potentially reconsidering and backtracking, in a scratchpad that's visually and structurally separate from its final response.

**Why this matters — the difference isn't cosmetic.** Manual CoT only ever gets a model to follow the reasoning *shape* you specified. Extended thinking lets the model discover its own reasoning path, self-correct mid-way, and allocate more or less effort depending on how hard the problem actually is. On genuinely hard multi-step problems (nontrivial math, multi-file debugging, complex planning), it reliably outperforms manual CoT prompting on the same base model.

**Where this shows up in real systems:**
- An agentic coding assistant deciding *how* to fix a bug across multiple files before touching anything.
- A financial-analysis agent reconciling numbers from several inconsistent sources before committing to a figure.
- **The 2 AM production bug this prevents:** a rate-limiter or payment-processing agent that has to reason through several interacting edge cases (partial failures, retries, idempotency) before deciding what action to take — the kind of task where a manually-prompted "think step by step" model plausibly skips a case, but a model given room to actually reason is more likely to catch it.

**Version note:** extended thinking shipped with Claude 3.7 (Feb 2025) as a toggleable native feature. Since then, **interleaved thinking** was added for Claude 4-generation models: the model can think *between* tool calls in an agentic loop too, not just once before its first response — reasoning about a tool's result before deciding what to do next, which is a meaningful chunk of what makes modern agents (including Claude Code) more reliable at long multi-step tasks.

### Using it

Pass a `thinking` parameter with a `budget_tokens` value (a soft cap on how many tokens the model may spend thinking). A few things to know:
- `max_tokens` must exceed `budget_tokens`, since `max_tokens` covers *both* the thinking block and the final answer.
- While thinking is enabled, you generally can't also set `temperature`, `top_p`, or `top_k` — the model controls its own sampling during reasoning.
- The response contains a `thinking` content block (its reasoning) followed by a `text` block (the actual answer). Only the `text` block is really meant to be shown to end users as "the answer" — the thinking block is genuinely useful for debugging and transparency, but it's a scratchpad, not a polished response.

Let's try it on a problem that tends to trip up models without room to reason.

In [ ]:
PROMPT = (
    "A farmer has chickens and rabbits. Together they have 35 heads and 94 legs. "
    "How many chickens and how many rabbits does the farmer have? Give me just the final numbers."
)

response = think(PROMPT, budget_tokens=2000)

for block in response.content:
    print(f"--- {block.type} ---")
    print(block.thinking if block.type == "thinking" else block.text)
    print()

Notice the `thinking` block works through the algebra (or a trial-and-error approach) before the `text` block states the final answer. You're seeing the model's actual reasoning process, not a reasoning-shaped performance you asked it to produce.

### Interleaved thinking in agentic loops

When Claude is calling tools in a loop (as in Appendix 10.2), interleaved thinking lets it reason about *each* tool result before deciding the next step — "the search returned three results, but only one is actually recent enough to trust, so I'll fetch that one before answering" — rather than reacting to tool output reflexively. This is enabled via a beta header and works alongside the same `tools` parameter you already know; see the docs for the current header name, since beta flags are the part of this API most likely to change.

---

## Exercise

### Exercise 10.5.1 - Separate the Scratchpad from the Answer

A common production mistake is showing a user the raw `thinking` block, which is unpolished and not meant to be user-facing. Write a function `get_final_answer(prompt, budget_tokens)` that calls Claude with extended thinking enabled and returns **only** the final text answer — no thinking content — as a string.

In [ ]:
# Your function goes here — this is the only field you should change
def get_final_answer(prompt, budget_tokens=2000):
    pass

answer = get_final_answer(
    "If a train leaves Boston at 60mph and another leaves New York at 40mph on the same track "
    "heading toward each other, and the cities are 200 miles apart, how many minutes until they meet? "
    "Give just the final number of minutes.",
    budget_tokens=1500,
)

print(answer)
print("\n--------------------------- GRADING ---------------------------")
print("Correct (should be close to 120):", "120" in answer)
print("No thinking leaked (shouldn't contain reasoning-style hedging):", "thinking" not in answer.lower())

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_5_1_hint; print(exercise_10_5_1_hint)

### Congrats!

**Recap:** extended thinking gives reasoning-capable models a native scratchpad, trained into the model rather than prompted into it — generally stronger than manual chain-of-thought on hard problems, at the cost of more tokens and latency. Manual CoT (Chapter 6) is still worth knowing: it's cheaper, works on every model, and gives you tighter control over exactly what steps get forced.

**Interview-answer framing:** "Extended thinking is a native reasoning mode where the model produces a separate thinking block, using reasoning learned during training, before its final answer — as opposed to manual chain-of-thought, where you're just asking the model to write out steps in its normal output. It's controlled via a `thinking` parameter with a token budget, and it meaningfully improves performance on hard multi-step problems like complex math, planning, and multi-file debugging. The tradeoff is cost and latency, so I'd reach for it on genuinely hard tasks and stick to manual prompting or a smaller model for simple ones."

Head over to Appendix 10.6 to learn about structured outputs.

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson.

In [ ]:
PROMPT = (
    "A farmer has chickens and rabbits. Together they have 35 heads and 94 legs. "
    "How many chickens and how many rabbits does the farmer have? Give me just the final numbers."
)

response = think(PROMPT, budget_tokens=2000)
for block in response.content:
    print(f"--- {block.type} ---")
    print(block.thinking if block.type == "thinking" else block.text)
    print()